# Лабораторная работа № 3
Есиков Сергей 

СПбАУ, 302 гр.

Вар 5

### Задание
![alt text](../tasks/3.png)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import t

In [ ]:
MY_VARIANT = 5
df = pd.read_csv('../data/3.csv', header=None, names=['variant', 'country', 'value'])

## 1. Выбрать данные своего варианта

In [ ]:
import itertools
import numpy as np
import pandas as pd
from scipy import stats

# 1. Загрузка данных (замените 'data.csv' на имя вашего файла)
# Предполагается, что в CSV два столбца: 'частица' и 'время_жизни'
try:
    df = pd.read_csv("data.csv")
except FileNotFoundError:
    # Пример для демонстрации, если файл не найден
    np.random.seed(42)
    particles = ["электрон", "мюон", "тау-лептон"]
    data = []
    for p, scale in zip(particles, [1e5, 2.2, 2.9]):
        data.extend(
            [
                {"частица": p, "время_жизни": x}
                for x in stats.expon.rvs(scale=scale, size=50)
            ]
        )
    df = pd.DataFrame(data)

# Переименуем столбцы для удобства, если они называются иначе
df.columns = ["particle", "lifetime"]

# 2. Вычисление доверительных интервалов (99%)
alpha = 0.01
intervals = {}

# Группируем по частицам
grouped = df.groupby("particle")["lifetime"]

print("--- Доверительные интервалы (99%) ---")
for particle, group in grouped:
    n = len(group)
    mean_x = group.mean()
    sum_x = group.sum()

    # Квантили хи-квадрат распределения
    chi2_low = stats.chi2.ppf(alpha / 2, df=2 * n)
    chi2_high = stats.chi2.ppf(1 - alpha / 2, df=2 * n)

    # Границы интервала для математического ожидания (1/lambda)
    ci_lower = (2 * sum_x) / chi2_high
    ci_upper = (2 * sum_x) / chi2_low

    intervals[particle] = (ci_lower, ci_upper)
    print(f"{particle}: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("\n--- Проверка пересечений ---")
# 3. Поиск пересекающихся пар
particle_names = list(intervals.keys())
intersecting_pairs = []
non_intersecting_pairs = []

for p1, p2 in itertools.combinations(particle_names, 2):
    ci1 = intervals[p1]
    ci2 = intervals[p2]

    # Условие пересечения двух интервалов [a, b] и [c, d]: max(a, c) <= min(b, d)
    if max(ci1[0], ci2[0]) <= min(ci1[1], ci2[1]):
        intersecting_pairs.append((p1, p2))
        print(f"Пересекаются: {p1} и {p2}")
    else:
        non_intersecting_pairs.append((p1, p2))
        print(f"НЕ пересекаются: {p1} и {p2}")
